# Encoder and Random Forest and larger datasets

So far we have been limiting ourselves to single subsets of the data. A specfic instrument, ionization mode and what not. In this case I want to take several datasets, train them with different encoders then use a union of the outputs to train a much larger Random forest (ideally balaning that RF).

## Import packages

In [118]:
# Import packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests

# from fcd_torch import FCD
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.decomposition import PCA

from fcd_torch import FCD
import rdkit

import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import functions_enc as f

## Globally used Functions

In [119]:
# Spectrum string to dataframe function
def spectrum_string_to_dataframe(df, spectrum_col, smiles_col):
    """
    Converts a DataFrame with a spectrum column (string of 'x:y' pairs) into a matrix
    where columns are unique x values, rows are spectra (even for duplicate SMILES), and values are y (intensity).
    The index will match the original DataFrame.
    """
    # Collect all unique x values (m/z)
    x_values_set = set()
    spectra_list = []
    for idx, row in df.iterrows():
        spectrum = row[spectrum_col]
        pairs = spectrum.split()
        xys = []
        for pair in pairs:
            try:
                x, y = pair.split(":") # Split into x and y
                #x = float(x.replace("'", "").replace('"', '')) # Remove quotes and convert to float (done in processing)
                #y = float(y.replace("'", "").replace('"', '')) # Remove quotes and convert to float (done in processing)
                xys.append((x, y))
                x_values_set.add(x)
            except Exception:
                continue
        spectra_list.append((row[smiles_col], dict(xys)))
    x_values = sorted(x_values_set) # Sort the x values to maintain order
    
    # Build the matrix
    matrix = []
    smiles_list = []
    for smiles, xy_dict in spectra_list:
        row = [xy_dict.get(x, 0.0) for x in x_values]
        matrix.append(row)
        smiles_list.append(smiles)
    df_matrix = pd.DataFrame(matrix, columns=[x for x in x_values]) # columns=[f"mz_{x}" for x in x_values]) to make stings
    df_matrix.insert(0, smiles_col, smiles_list)
    df_matrix.index = df.index  # preserve original row order/index
    return df_matrix

In [120]:
# Binning functions
# Binning the spectra data 
def bin_spectra_by_integer_mz(df):
    """
    Bins the spectra data by rounding m/z (column names) to the nearest integer,
    then sums intensities for duplicate integer bins.
    Assumes first column is SMILES, rest are m/z columns (floats).
    """
    smiles_col = df.columns[0]
    spectra = df.iloc[:, 1:]
    # Map each column to its integer bin
    int_mz = [int(round(float(c))) for c in spectra.columns]
    spectra.columns = int_mz
    # Group columns by integer m/z and sum
    binned = spectra.groupby(level=0, axis=1).sum()
    # Add the SMILES column back
    binned.insert(0, smiles_col, df[smiles_col])
    return binned

# Fill in the missing integer columns 
def fill_missing_integer_columns(df):
    """
    Ensures all integer columns from 1 to the maximum are present in the DataFrame.
    Missing columns are filled with zeros.
    Assumes the first column is the label (e.g., SMILES).
    """
    # Get the integer columns (skip the first column)
    int_cols = [col for col in df.columns[1:] if isinstance(col, int)]
    #min_col = min(int_cols)
    max_col = max(int_cols)
    all_int_cols = list(range(1, max_col + 1))
    # Find missing columns
    missing_cols = set(all_int_cols) - set(int_cols)
    # Add missing columns with zeros
    for col in missing_cols:
        df[col] = 0.0
    # Reorder columns: first column, then sorted integer columns
    ordered_cols = [df.columns[0]] + sorted(all_int_cols)
    df = df[ordered_cols]
    return df

In [121]:
# Cate's smiles to ChemNet embedding code
def get_chemnet_emb_from_smiles(smiles_list):
    """
    Get ChemNet embeddings from a list of SMILES strings.

    Parameters:
    smiles_list (list): List of SMILES strings.

    Returns:
    dict: A dictionary mapping each SMILES string to its corresponding ChemNet embedding.
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    fcd = FCD(device, n_jobs=1)
    
    smiles_emb_dict = {}

    for smiles in smiles_list:
        try:
            emb = fcd.get_predictions([smiles])[0]
            smiles_emb_dict[smiles] = list(emb)
        except KeyError as e:
            if e == 'PropertyTable':
                smiles_emb_dict[smiles] = 'unknown'

    return smiles_emb_dict

In [122]:
def create_dataset_tensors(spectra_dataset, embedding_df, device, start_idx=None, stop_idx=None):
    """
    Create tensors from the provided spectra dataset and embedding DataFrame.

    Parameters:
    ----------
    spectra_dataset : pd.DataFrame
        DataFrame containing spectral data and chemical labels. Assumes specific 
        columns for processing based on the `carl` flag.

    embedding_df : pd.DataFrame
        DataFrame containing embeddings for chemicals, with 'Embedding Floats' 
        column corresponding to ChemNet embeddings.

    device : torch.device
        The device (CPU or GPU) on which to store the tensors.

    carl : bool, optional
        If True, processes the dataset assuming it has a different structure 
        (specifically without an 'Unnamed: 0' column). Default is False.

    Returns:
    -------
    tuple
        A tuple containing:
        - embeddings_tensor (torch.Tensor): Tensor of true embeddings for the chemicals.
        - spectra_tensor (torch.Tensor): Tensor of spectral data.
        - chem_encodings_tensor (torch.Tensor): Tensor of chemical name encodings.
        - spectra_indices_tensor (torch.Tensor): Tensor of indices corresponding to the spectra.
    """
    spectra = spectra_dataset.iloc[:,1:-1]

    #chem_encodings = spectra_dataset.iloc[:,-8:]

    # create tensors of spectra, true embeddings, and chemical name encodings for train and val
    chem_labels = list(spectra_dataset['SMILES_spectra'])
    embeddings_tensor = torch.Tensor([embedding_df.loc[embedding_df['SMILES'] == chem_name].iloc[0, 1:].values.astype(float) for chem_name in chem_labels]).to(device)
    spectra_tensor = torch.Tensor(spectra.values).to(device)
    #chem_encodings_tensor = torch.Tensor(chem_encodings.values).to(device)
    spectra_indices_tensor = torch.Tensor(spectra_dataset['index'].to_numpy()).to(device)

    return embeddings_tensor, spectra_tensor, spectra_indices_tensor #, chem_encodings_tensor

## Data Processing: Upload, Edits, and Split

In [123]:
# The 5/20 dataset with rat based toxicity data
df3 = pd.read_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/MIT_LL_data3.csv")
print(df3.shape)
df3.head()


# Uniformity of ionization model labels
print(df3["Ionization_Mode"].unique())
df3["Ionization_Mode"] = df3["Ionization_Mode"].replace("'Positive'", "'positive'")
print(df3["Ionization_Mode"].unique())

# Remove the N/A values in Ionization_Mode
df3 = df3[df3["Ionization_Mode"] != "'NaN'"]
print(df3["Ionization_Mode"].unique())

# Removing the '' from the SMILES
# Remove single quotes from all string columns in df3
df3 = df3.applymap(lambda x: x.replace("'", "") if isinstance(x, str) else x)
#df3["SMILES_spectra"] = df3["SMILES_spectra"].str.replace("'", "")

# Data split based on the 'Group' column
df3_QQpos = df3[df3['Group'] == 'Q-Orbitrap-positive'] # 1307
df3_QQneg = df3[df3['Group'] == 'Q-Orbitrap-negative'] # 756
df3_QTOFpos = df3[df3['Group'] == 'Q-TOF-positive'] # 736  
df3_LTQOpos = df3[df3['Group'] == 'LTQ-Orbitrap-positive'] # 481 

# With these there may not be enough data to train a model
df3_QQQpos = df3[df3['Group'] == 'QQQ-positive'] # 253
df3_QTOFneg = df3[df3['Group'] == 'Q-TOF-negative'] # 188
df3_QQQneg = df3[df3['Group'] == 'QQQ-negative'] # 85
df3_Opos = df3[df3['Group'] == 'Other-positive'] # 71
df3_LTQOneg = df3[df3['Group'] == 'LTQ-Orbitrap-negative'] # 63
df3_LTQpos = df3[df3['Group'] == 'LTQ-positive'] # 19
df3_QQQnan = df3[df3['Group'] == 'QQQ-nan'] # 18
df3_Oneg = df3[df3['Group'] == 'Other-negative'] # 13
df3_LTQneg = df3[df3['Group'] == 'LTQ-negative'] # 11

(4001, 16)
["'positive'" "'negative'" "'Positive'" "'NaN'"]
["'positive'" "'negative'" "'NaN'"]
["'positive'" "'negative'"]


## Data Processing: Binning

In [124]:
df3_QQpos_matrix = spectrum_string_to_dataframe(df3_QQpos,'Spectrum', 'SMILES_spectra')
df3_QQpos_spectra = df3_QQpos_matrix
df3_QQneg_matrix = spectrum_string_to_dataframe(df3_QQneg,'Spectrum', 'SMILES_spectra')
df3_QQneg_spectra = df3_QQneg_matrix
df3_QTOFpos_matrix = spectrum_string_to_dataframe(df3_QTOFpos,'Spectrum', 'SMILES_spectra')
df3_QTOFpos_spectra = df3_QTOFpos_matrix
df3_LTQOpos_matrix = spectrum_string_to_dataframe(df3_LTQOpos,'Spectrum', 'SMILES_spectra')
df3_LTQOpos_spectra = df3_LTQOpos_matrix

In [125]:
# Processing of the spectra to enable binning to work (consider changing this to a function)

# Conversion of m/z values to floats, keeping the first column (SMIELS) as is
# For df3_QQpos_spectra
cols = df3_QQpos_spectra.columns.tolist()
new_cols = [cols[0]] + [float(c) for c in cols[1:]]
df3_QQpos_spectra.columns = new_cols

# For df3_QQneg_spectra
cols = df3_QQneg_spectra.columns.tolist()
new_cols = [cols[0]] + [float(c) for c in cols[1:]]
df3_QQneg_spectra.columns = new_cols

# For df3_QTOFpos_spectra
cols = df3_QTOFpos_spectra.columns.tolist()
new_cols = [cols[0]] + [float(c) for c in cols[1:]]
df3_QTOFpos_spectra.columns = new_cols

# For df3_LTQOpos_spectra
cols = df3_LTQOpos_spectra.columns.tolist()
new_cols = [cols[0]] + [float(c) for c in cols[1:]]
df3_LTQOpos_spectra.columns = new_cols


# Convert all elements except the first column to float
df3_QQpos_spectra.iloc[:, 1:] = df3_QQpos_spectra.iloc[:, 1:].apply(pd.to_numeric, errors='coerce')
df3_QQneg_spectra.iloc[:, 1:] = df3_QQneg_spectra.iloc[:, 1:].apply(pd.to_numeric, errors='coerce')
df3_QTOFpos_spectra.iloc[:, 1:] = df3_QTOFpos_spectra.iloc[:, 1:].apply(pd.to_numeric, errors='coerce')
df3_LTQOpos_spectra.iloc[:, 1:] = df3_LTQOpos_spectra.iloc[:, 1:].apply(pd.to_numeric, errors='coerce')

all_float = all(isinstance(c, float) for c in df3_QQneg_spectra.columns[1:])
print("All columns are float:", all_float)
all_float = all(isinstance(c, float) for c in df3_QQpos_spectra.columns[1:])
print("All columns are float:", all_float)
all_float = all(isinstance(c, float) for c in df3_QTOFpos_spectra.columns[1:])
print("All columns are float:", all_float)
all_float = all(isinstance(c, float) for c in df3_LTQOpos_spectra.columns[1:])
print("All columns are float:", all_float)

# Check if every element is a float
spectra = df3_QQpos_spectra.iloc[:, 1:]
all_float_elements = spectra.applymap(lambda x: isinstance(x, float)).all().all()
print("All elements are float:", all_float_elements)
spectra = df3_QQneg_spectra.iloc[:, 1:]
all_float_elements = spectra.applymap(lambda x: isinstance(x, float)).all().all()
print("All elements are float:", all_float_elements)
spectra = df3_QTOFpos_spectra.iloc[:, 1:]
all_float_elements = spectra.applymap(lambda x: isinstance(x, float)).all().all()
print("All elements are float:", all_float_elements)
spectra = df3_LTQOpos_spectra.iloc[:, 1:]
all_float_elements = spectra.applymap(lambda x: isinstance(x, float)).all().all()
print("All elements are float:", all_float_elements)

# Sort columns by float value, keep the first column (SMILES) first
cols = df3_QQpos_spectra.columns.tolist()
sorted_cols = [cols[0]] + sorted(cols[1:], key=float)
df3_QQpos_spectra = df3_QQpos_spectra[sorted_cols]
cols = df3_QQneg_spectra.columns.tolist()
sorted_cols = [cols[0]] + sorted(cols[1:], key=float)
df3_QQneg_spectra = df3_QQneg_spectra[sorted_cols]
cols = df3_QTOFpos_spectra.columns.tolist()
sorted_cols = [cols[0]] + sorted(cols[1:], key=float)
df3_QTOFpos_spectra = df3_QTOFpos_spectra[sorted_cols]
cols = df3_LTQOpos_spectra.columns.tolist()
sorted_cols = [cols[0]] + sorted(cols[1:], key=float)
df3_LTQOpos_spectra = df3_LTQOpos_spectra[sorted_cols]

All columns are float: True
All columns are float: True
All columns are float: True
All columns are float: True
All elements are float: True
All elements are float: True
All elements are float: True
All elements are float: True


In [126]:
df3_QTOFpos_spectra.head()

,SMILES_spectra,29.0112,30.032,30.0323,31.01686,31.54035,38.5076,39.0214,39.0215,39.02194,...,1879.890381,1881.465942,1890.796509,1894.779541,1939.240845,1965.805054,1966.380615,1982.848389,2000.461914,2000.942627
30,C1=CN(C=N1)C(C1=CC=CC=C1)C1=CC=C(C=C1)C1=CC=CC=C1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
31,C1=CN(C=N1)C(C1=CC=CC=C1)C1=CC=C(C=C1)C1=CC=CC=C1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
32,C1=CN(C=N1)C(C1=CC=CC=C1)C1=CC=C(C=C1)C1=CC=CC=C1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
33,C1=CN(C=N1)C(C1=CC=CC=C1)C1=CC=C(C=C1)C1=CC=CC=C1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
34,C1=CN(C=N1)C(C1=CC=CC=C1)C1=CC=C(C=C1)C1=CC=CC=C1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [127]:
# Use the binning function 
# On df3_QQpos_spectra
binned_df3_QQpos_spectra = bin_spectra_by_integer_mz(df3_QQpos_spectra)
binned_df3_QQpos_spectra_filled = fill_missing_integer_columns(binned_df3_QQpos_spectra)

# On df3_QQpos_spectra
binned_df3_QQneg_spectra = bin_spectra_by_integer_mz(df3_QQneg_spectra)
binned_df3_QQneg_spectra_filled = fill_missing_integer_columns(binned_df3_QQneg_spectra)

# On df3_QQpos_spectra
binned_df3_QTOFpos_spectra = bin_spectra_by_integer_mz(df3_QTOFpos_spectra)
binned_df3_QTOFpos_spectra_filled = fill_missing_integer_columns(binned_df3_QTOFpos_spectra)

# On df3_LTQOpos_spectra
binned_df3_LTQOpos_spectra = bin_spectra_by_integer_mz(df3_LTQOpos_spectra)
binned_df3_LTQOpos_spectra_filled = fill_missing_integer_columns(binned_df3_LTQOpos_spectra)

/tmp/ipykernel_115872/339873635.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  binned.insert(0, smiles_col, df[smiles_col])
/tmp/ipykernel_115872/339873635.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col] = 0.0
/tmp/ipykernel_115872/339873635.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = fra

In [128]:
print(binned_df3_QQpos_spectra_filled.shape)
print(binned_df3_QQneg_spectra_filled.shape)
print(binned_df3_QTOFpos_spectra_filled.shape)
print(binned_df3_LTQOpos_spectra_filled.shape)

(1307, 680)
(756, 493)
(736, 2002)
(481, 816)


In [129]:
# # Save the binned spectra to a CSV file
# binned_df3_QQpos_spectra_filled.to_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/binned_df3_QQpos_spectra.csv", index=False)
# binned_df3_QQneg_spectra_filled.to_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/binned_df3_QQneg_spectra.csv", index=False)
# binned_df3_QTOFpos_spectra_filled.to_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/binned_df3_QTOFpos_spectra.csv", index=False)
# binned_df3_LTQOpos_spectra_filled.to_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/binned_df3_LTQOpos_spectra.csv", index=False)

## ChemNet True Embeddings

In [130]:
ChemNet_of_df3_QQpos_no_repeats_dict = get_chemnet_emb_from_smiles(df3_QQpos_spectra["SMILES_spectra"])
ChemNet_of_df3_QQneg_no_repeats_dict = get_chemnet_emb_from_smiles(df3_QQneg_spectra["SMILES_spectra"])
ChemNet_of_df3_QTOFpos_no_repeats_dict = get_chemnet_emb_from_smiles(df3_QTOFpos_spectra["SMILES_spectra"])
ChemNet_of_df3_LTQOpos_no_repeats_dict = get_chemnet_emb_from_smiles(df3_LTQOpos_spectra["SMILES_spectra"])

/home/dlipsey/.local/lib/python3.8/site-packages/fcd_torch/fcd.py:51: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  keras_config = torch.load(model_path)


In [131]:
# Making the ChemNet embeddings into a dataframes
# For df3_QQpos
ChemNet_of_df3_QQpos_no_repeats = pd.DataFrame.from_dict(ChemNet_of_df3_QQpos_no_repeats_dict, orient='index')
ChemNet_of_df3_QQpos_no_repeats.reset_index(inplace=True)
ChemNet_of_df3_QQpos_no_repeats.rename(columns={'index': 'SMILES'}, inplace=True)

# For df3_QQneg
ChemNet_of_df3_QQneg_no_repeats = pd.DataFrame.from_dict(ChemNet_of_df3_QQneg_no_repeats_dict, orient='index')
ChemNet_of_df3_QQneg_no_repeats.reset_index(inplace=True)
ChemNet_of_df3_QQneg_no_repeats.rename(columns={'index': 'SMILES'}, inplace=True)

# For df3_QTOFpos
ChemNet_of_df3_QTOFpos_no_repeats = pd.DataFrame.from_dict(ChemNet_of_df3_QTOFpos_no_repeats_dict, orient='index')
ChemNet_of_df3_QTOFpos_no_repeats.reset_index(inplace=True)
ChemNet_of_df3_QTOFpos_no_repeats.rename(columns={'index': 'SMILES'}, inplace=True)

# For df3_LTQOpos
ChemNet_of_df3_LTQOpos_no_repeats = pd.DataFrame.from_dict(ChemNet_of_df3_LTQOpos_no_repeats_dict, orient='index')
ChemNet_of_df3_LTQOpos_no_repeats.reset_index(inplace=True)
ChemNet_of_df3_LTQOpos_no_repeats.rename(columns={'index': 'SMILES'}, inplace=True)

In [132]:
# # Save ChemNet embeddings to a CSV file
# ChemNet_of_df3_QQpos_no_repeats.to_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/ChemNet_of_df3_QQpos_no_repeats.csv", index=False)
# ChemNet_of_df3_QQneg_no_repeats.to_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/ChemNet_of_df3_QQneg_no_repeats.csv", index=False)
# ChemNet_of_df3_QTOFpos_no_repeats.to_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/ChemNet_of_df3_QTOFpos_no_repeats.csv", index=False)
# ChemNet_of_df3_LTQOpos_no_repeats.to_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/ChemNet_of_df3_LTQOpos_no_repeats.csv", index=False)

## Encoder Implementation

### Encoder architecture

In [133]:
batch_size = 64
epochs=500
lr=0.0001
criterion=nn.MSELoss()
output_size = 512
num_layers = 10

#%%
# Encoder architecture (With Validation Set)
class Encoder(nn.Module):
    def __init__(self, input_size, output_size, num_layers):
        super().__init__()
        layers = []
        layer_sizes = np.linspace(input_size, output_size, num_layers + 1, dtype=int)
        for i in range(num_layers):
            layers.append(nn.Linear(layer_sizes[i], layer_sizes[i+1]))
            if i < num_layers - 1:
                layers.append(nn.LeakyReLU(inplace=True))
        self.encoder = nn.Sequential(*layers)

    def forward(self, x):
        return self.encoder(x)

def train_model(model, train_data, val_data, epochs, learning_rate, criterion, device):
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for batch, true_embeddings, _ in train_data:
            batch = batch.to(device)
            true_embeddings = true_embeddings.to(device)

            optimizer.zero_grad()
            batch_predicted_embeddings = model(batch)
            loss = criterion(batch_predicted_embeddings, true_embeddings)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        average_train_loss = running_loss / len(train_loader)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for val_batch, val_true_embeddings, _ in val_data:
                val_batch = val_batch.to(device)
                val_true_embeddings = val_true_embeddings.to(device)

                val_batch_predicted_embeddings = model(val_batch)

                val_loss = criterion(val_batch_predicted_embeddings, val_true_embeddings)
                val_loss += loss.item()
        average_val_loss = val_loss / len(val_loader)

        print(f'Epoch [{epoch+1}/{epochs}]')
        print(f'   Training loss: {average_train_loss}')
        print(f'   Validation loss: {average_val_loss}')

    return model
#%%


## Implementation of the encoder on df3_QQpos

In [134]:
# Training and validation dataset split for the positive ionization mode
df3_QQpos_spectra= pd.read_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/binned_df3_QQpos_spectra.csv")
name_smiles_embedding_df = pd.read_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/ChemNet_of_df3_QQpos_no_repeats.csv")
# Generalize the syntax
dataset =  df3_QQpos_spectra

# Count occurrences of each SMILES_spectra and keep only SMILES_spectra with at least 4 entries
counts = dataset['SMILES_spectra'].value_counts()
valid_smiles = counts[counts >= 4].index
filtered_dataset = dataset[dataset['SMILES_spectra'].isin(valid_smiles)].copy()

train_indices = []
test_indices = []

for smiles, group in filtered_dataset.groupby('SMILES_spectra'):
    idx = group.index.tolist()
    n = len(idx)
    np.random.shuffle(idx)  # shuffle for randomness
    split = n // 2
    test_idx = idx[:split]
    train_idx = idx[split:]
    # If odd, train gets the extra
    train_indices.extend(train_idx)
    test_indices.extend(test_idx)

train_data = filtered_dataset.loc[train_indices].reset_index(drop=True)
test_data = filtered_dataset.loc[test_indices].reset_index(drop=True)

# Add an 'index' column 
train_data['index'] = train_data.index
test_data['index'] = test_data.index

print(train_data.shape)
print(test_data.shape)

# Make a copy
train_data_df3_QQpos = train_data.copy()

# Make a copy
train_data_df3_QQneg = train_data.copy()
# Load val_data under the right name
val_data = test_data

# # train_data = pd.read_csv("/") 
# # val_data = pd.read_csv(" ") 



(609, 681)
(579, 681)


In [135]:
# Encoder training
device = f.set_up_gpu()
# device = torch.device('cpu')

# Training set
y_train, x_train, train_indices_tensor = create_dataset_tensors(
    train_data, name_smiles_embedding_df, device, start_idx=2, stop_idx=-0)
#sorted_chem_names = list(train_data.columns[-8:])
del train_data

# Validation set
y_val, x_val, val_indices_tensor = create_dataset_tensors(
    val_data, name_smiles_embedding_df, device, start_idx=2, stop_idx=-0)
del val_data

train_data = TensorDataset(x_train, y_train, train_indices_tensor)
val_data = TensorDataset(x_val, y_val, val_indices_tensor)
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
#%%
encoder_df3_QQpos = Encoder(input_size=x_train.shape[1], output_size=output_size, num_layers=num_layers).to(device)

#%%
model_df3_QQpos = train_model(
    model=encoder_df3_QQpos,
    train_data=train_loader,
    val_data=val_loader,
    epochs=epochs,
    learning_rate=lr,
    criterion=criterion,
    device=device
)

Selected GPU ID: 1
  Name: NVIDIA GeForce RTX 4090
  Memory Free: 22760.0 MB
  Memory Used: 1457.0 MB
  GPU Load: 0.00%
Current device ID:  cuda:1
PyTorch current device ID: 1
PyTorch current device name: NVIDIA GeForce RTX 4090
Epoch [1/500]
   Training loss: 0.08984625414013862
   Validation loss: 0.015067157335579395
Epoch [2/500]
   Training loss: 0.08274132013320923
   Validation loss: 0.013616256415843964
Epoch [3/500]
   Training loss: 0.07235830500721932
   Validation loss: 0.012838325463235378
Epoch [4/500]
   Training loss: 0.0660598162561655
   Validation loss: 0.011486870236694813
Epoch [5/500]
   Training loss: 0.06420949138700963
   Validation loss: 0.011278827674686909
Epoch [6/500]
   Training loss: 0.06393651403486729
   Validation loss: 0.011615610681474209
Epoch [7/500]
   Training loss: 0.06336918622255325
   Validation loss: 0.01096972543746233
Epoch [8/500]
   Training loss: 0.0633424673229456
   Validation loss: 0.011222898960113525
Epoch [9/500]
   Training loss

## Implementation of the encoder on df3_QQneg

In [136]:
# Training and validation dataset split 
df3_QQneg_spectra= pd.read_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/binned_df3_QQneg_spectra.csv")
name_smiles_embedding_df = pd.read_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/ChemNet_of_df3_QQneg_no_repeats.csv")

# Generalize the syntax
dataset =  df3_QQneg_spectra

# Count occurrences of each SMILES_spectra and keep only SMILES_spectra with at least 4 entries
counts = dataset['SMILES_spectra'].value_counts()
valid_smiles = counts[counts >= 4].index
filtered_dataset = dataset[dataset['SMILES_spectra'].isin(valid_smiles)].copy()

train_indices = []
test_indices = []

for smiles, group in filtered_dataset.groupby('SMILES_spectra'):
    idx = group.index.tolist()
    n = len(idx)
    np.random.shuffle(idx)  # shuffle for randomness
    split = n // 2
    test_idx = idx[:split]
    train_idx = idx[split:]
    # If odd, train gets the extra
    train_indices.extend(train_idx)
    test_indices.extend(test_idx)

train_data = filtered_dataset.loc[train_indices].reset_index(drop=True)
test_data = filtered_dataset.loc[test_indices].reset_index(drop=True)

# Add an 'index' column 
train_data['index'] = train_data.index
test_data['index'] = test_data.index

print(train_data.shape)
print(test_data.shape)

# Make a copy
train_data_df3_QQneg = train_data.copy()

# Make a copy
train_data_df3_QQneg = train_data.copy()
# Load val_data under the right name
val_data = test_data

# # train_data = pd.read_csv("/") 
# # val_data = pd.read_csv(" ") 

(359, 494)
(342, 494)


In [137]:
# Encoder training
device = f.set_up_gpu()
# device = torch.device('cpu')

# Training set
y_train, x_train, train_indices_tensor = create_dataset_tensors(
    train_data, name_smiles_embedding_df, device, start_idx=2, stop_idx=-0)
#sorted_chem_names = list(train_data.columns[-8:])
del train_data

# Validation set
y_val, x_val, val_indices_tensor = create_dataset_tensors(
    val_data, name_smiles_embedding_df, device, start_idx=2, stop_idx=-0)
del val_data

train_data = TensorDataset(x_train, y_train, train_indices_tensor)
val_data = TensorDataset(x_val, y_val, val_indices_tensor)
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
#%%
encoder_df3_QQneg = Encoder(input_size=x_train.shape[1], output_size=output_size, num_layers=num_layers).to(device)

#%%
model_df3_QQneg = train_model(
    model=encoder_df3_QQneg,
    train_data=train_loader,
    val_data=val_loader,
    epochs=epochs,
    learning_rate=lr,
    criterion=criterion,
    device=device
)

Selected GPU ID: 1
  Name: NVIDIA GeForce RTX 4090
  Memory Free: 22760.0 MB
  Memory Used: 1457.0 MB
  GPU Load: 0.00%
Current device ID:  cuda:1
PyTorch current device ID: 1
PyTorch current device name: NVIDIA GeForce RTX 4090
Epoch [1/500]
   Training loss: 0.09160873045523961
   Validation loss: 0.03009681962430477
Epoch [2/500]
   Training loss: 0.08963835860292117
   Validation loss: 0.029535075649619102
Epoch [3/500]
   Training loss: 0.0843353954454263
   Validation loss: 0.026874039322137833
Epoch [4/500]
   Training loss: 0.0757488173743089
   Validation loss: 0.024647703394293785
Epoch [5/500]
   Training loss: 0.0697996790210406
   Validation loss: 0.02257016859948635
Epoch [6/500]
   Training loss: 0.06557077417771022
   Validation loss: 0.02095988020300865
Epoch [7/500]
   Training loss: 0.06383562708894412
   Validation loss: 0.021459240466356277
Epoch [8/500]
   Training loss: 0.06247100979089737
   Validation loss: 0.02090494893491268
Epoch [9/500]
   Training loss: 0.

## Implementation of the encoder on df3_QTOFpos

In [138]:
# Training and validation dataset split 
df3_QTOFpos_spectra= pd.read_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/binned_df3_QTOFpos_spectra.csv")
name_smiles_embedding_df = pd.read_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/ChemNet_of_df3_QTOFpos_no_repeats.csv")

# Generalize the syntax
dataset =  df3_QTOFpos_spectra

# Count occurrences of each SMILES_spectra and keep only SMILES_spectra with at least 4 entries
counts = dataset['SMILES_spectra'].value_counts()
valid_smiles = counts[counts >= 4].index
filtered_dataset = dataset[dataset['SMILES_spectra'].isin(valid_smiles)].copy()

train_indices = []
test_indices = []

for smiles, group in filtered_dataset.groupby('SMILES_spectra'):
    idx = group.index.tolist()
    n = len(idx)
    np.random.shuffle(idx)  # shuffle for randomness
    split = n // 2
    test_idx = idx[:split]
    train_idx = idx[split:]
    # If odd, train gets the extra
    train_indices.extend(train_idx)
    test_indices.extend(test_idx)

train_data = filtered_dataset.loc[train_indices].reset_index(drop=True)
test_data = filtered_dataset.loc[test_indices].reset_index(drop=True)

# Add an 'index' column 
train_data['index'] = train_data.index
test_data['index'] = test_data.index

print(train_data.shape)
print(test_data.shape)

# Make a copy
train_data_df3_QQneg = train_data.copy()

# Make a copy
train_data_df3_QQneg = train_data.copy()
# Load val_data under the right name
val_data = test_data

# # train_data = pd.read_csv("/") 
# # val_data = pd.read_csv(" ") 


(330, 2003)
(308, 2003)


In [139]:
# Encoder training
device = f.set_up_gpu()
# device = torch.device('cpu')

# Training set
y_train, x_train, train_indices_tensor = create_dataset_tensors(
    train_data, name_smiles_embedding_df, device, start_idx=2, stop_idx=-0)
#sorted_chem_names = list(train_data.columns[-8:])
del train_data

# Validation set
y_val, x_val, val_indices_tensor = create_dataset_tensors(
    val_data, name_smiles_embedding_df, device, start_idx=2, stop_idx=-0)
del val_data

train_data = TensorDataset(x_train, y_train, train_indices_tensor)
val_data = TensorDataset(x_val, y_val, val_indices_tensor)
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
#%%
encoder_df3_QTOFpos = Encoder(input_size=x_train.shape[1], output_size=output_size, num_layers=num_layers).to(device)

#%%
model_df3_QTOFpos = train_model(
    model=encoder_df3_QTOFpos,
    train_data=train_loader,
    val_data=val_loader,
    epochs=epochs,
    learning_rate=lr,
    criterion=criterion,
    device=device
)

Selected GPU ID: 1
  Name: NVIDIA GeForce RTX 4090
  Memory Free: 22714.0 MB
  Memory Used: 1503.0 MB
  GPU Load: 0.00%
Current device ID:  cuda:1
PyTorch current device ID: 1
PyTorch current device name: NVIDIA GeForce RTX 4090
Epoch [1/500]
   Training loss: 0.08783312514424324
   Validation loss: 0.030452316626906395
Epoch [2/500]
   Training loss: 0.08017526318629582
   Validation loss: 0.027255762368440628
Epoch [3/500]
   Training loss: 0.07056530751287937
   Validation loss: 0.023154664784669876
Epoch [4/500]
   Training loss: 0.06563500811656316
   Validation loss: 0.022196579724550247
Epoch [5/500]
   Training loss: 0.06384011171758175
   Validation loss: 0.022777589038014412
Epoch [6/500]
   Training loss: 0.06313613181312878
   Validation loss: 0.022907409816980362
Epoch [7/500]
   Training loss: 0.062013200794657074
   Validation loss: 0.021942049264907837
Epoch [8/500]
   Training loss: 0.06204786834617456
   Validation loss: 0.022208988666534424
Epoch [9/500]
   Training 

## Implementation of the encoder on df3_LTQOpos

In [140]:
# Training and validation dataset split 
df3_LTQOpos_spectra= pd.read_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/binned_df3_LTQOpos_spectra.csv")
name_smiles_embedding_df = pd.read_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/ChemNet_of_df3_LTQOpos_no_repeats.csv")

# Generalize the syntax
dataset = df3_LTQOpos_spectra

# Count occurrences of each SMILES_spectra and keep only SMILES_spectra with at least 4 entries
counts = dataset['SMILES_spectra'].value_counts()
valid_smiles = counts[counts >= 4].index
filtered_dataset = dataset[dataset['SMILES_spectra'].isin(valid_smiles)].copy()

train_indices = []
test_indices = []

for smiles, group in filtered_dataset.groupby('SMILES_spectra'):
    idx = group.index.tolist()
    n = len(idx)
    np.random.shuffle(idx)  # shuffle for randomness
    split = n // 2
    test_idx = idx[:split]
    train_idx = idx[split:]
    # If odd, train gets the extra
    train_indices.extend(train_idx)
    test_indices.extend(test_idx)

train_data = filtered_dataset.loc[train_indices].reset_index(drop=True)
test_data = filtered_dataset.loc[test_indices].reset_index(drop=True)

# Add an 'index' column 
train_data['index'] = train_data.index
test_data['index'] = test_data.index

print(train_data.shape)
print(test_data.shape)

# Make a copy
train_data_df3_QQneg = train_data.copy()
# Load val_data under the right name
val_data = test_data

# # train_data = pd.read_csv("/") 
# # val_data = pd.read_csv(" ") 


(239, 817)
(233, 817)


In [141]:
# Encoder training
device = f.set_up_gpu()
# device = torch.device('cpu')

# Training set
y_train, x_train, train_indices_tensor = create_dataset_tensors(
    train_data, name_smiles_embedding_df, device, start_idx=2, stop_idx=-0)
#sorted_chem_names = list(train_data.columns[-8:])
del train_data

# Validation set
y_val, x_val, val_indices_tensor = create_dataset_tensors(
    val_data, name_smiles_embedding_df, device, start_idx=2, stop_idx=-0)
del val_data

train_data = TensorDataset(x_train, y_train, train_indices_tensor)
val_data = TensorDataset(x_val, y_val, val_indices_tensor)
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
#%%
encoder_df3_LTQOpos = Encoder(input_size=x_train.shape[1], output_size=output_size, num_layers=num_layers).to(device)

#%%
model_df3_LTQOpos = train_model(
    model=encoder_df3_LTQOpos,
    train_data=train_loader,
    val_data=val_loader,
    epochs=epochs,
    learning_rate=lr,
    criterion=criterion,
    device=device
)

Selected GPU ID: 1
  Name: NVIDIA GeForce RTX 4090
  Memory Free: 22446.0 MB
  Memory Used: 1771.0 MB
  GPU Load: 0.00%
Current device ID:  cuda:1
PyTorch current device ID: 1
PyTorch current device name: NVIDIA GeForce RTX 4090
Epoch [1/500]
   Training loss: 0.08724313043057919
   Validation loss: 0.03947455435991287
Epoch [2/500]
   Training loss: 0.08596738614141941
   Validation loss: 0.03927740827202797
Epoch [3/500]
   Training loss: 0.08274242281913757
   Validation loss: 0.036261335015296936
Epoch [4/500]
   Training loss: 0.07701053656637669
   Validation loss: 0.034671686589717865
Epoch [5/500]
   Training loss: 0.07291135564446449
   Validation loss: 0.03344455361366272
Epoch [6/500]
   Training loss: 0.06800639443099499
   Validation loss: 0.030196525156497955
Epoch [7/500]
   Training loss: 0.06467895954847336
   Validation loss: 0.029130052775144577
Epoch [8/500]
   Training loss: 0.062002234160900116
   Validation loss: 0.027001798152923584
Epoch [9/500]
   Training los

## Encoder Testing

In [142]:
# Encoder excecution df3_QQpos
encoder_df3_QQpos.eval()
with torch.no_grad():
    test_output = encoder_df3_QQpos(x_val)  # output.shape will be (batch_size, 512)

with torch.no_grad():
    train_output = encoder_df3_QQpos(x_train)  # output.shape will be (batch_size, 512)

# Encoder excecution df3_QQneg
encoder_df3_QQneg.eval()
with torch.no_grad():
    test_output = encoder_df3_QQneg(x_val)  # output.shape will be (batch_size, 512)

with torch.no_grad():
    train_output = encoder_df3_QQneg(x_train)  # output.shape will be (batch_size, 512)

# Encoder excecution df3_QTOFpos
encoder_df3_QTOFpos.eval()
with torch.no_grad():
    test_output = encoder_df3_QTOFpos(x_val)  # output.shape will be (batch_size, 512)

with torch.no_grad():
    train_output = encoder_df3_QTOFpos(x_train)  # output.shape will be (batch_size, 512)

# Encoder excecution df3_LTQOpos
encoder_df3_LTQOpos.eval()
with torch.no_grad():
    test_output = encoder_df3_LTQOpos(x_val)  # output.shape will be (batch_size, 512)

with torch.no_grad():
    train_output = encoder_df3_LTQOpos(x_train)  # output.shape will be (batch_size, 512)

RuntimeError: mat1 and mat2 shapes cannot be multiplied (233x815 and 679x662)